In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
# 导入 HuggingFace transformers 官方实现:AutoModelForCausalLM 用于加载 Gemma3 因果语言模型,AutoTokenizer 用于加载对应分词器

# 使用 Google 官方发布的 Gemma3 270M 指令微调(it = instruction-tuned)版本作为参考实现的权重来源
model_id = "google/gemma-3-270m-it"
# 测试用的输入提示语,用于生成参考输出以校验自实现的 Gemma3 是否正确
prompt = "Give me a short introduction to large language models."

# 从 HuggingFace Hub 下载并加载与 model_id 对应的分词器(tokenizer)
tokenizer = AutoTokenizer.from_pretrained(model_id)
# 从 HuggingFace Hub 下载并加载官方 Gemma3 因果语言模型权重,作为对照基准
model = AutoModelForCausalLM.from_pretrained(model_id)
# 关闭采样(sampling),改为确定性的贪婪解码(greedy decoding),保证输出可复现,便于与自实现结果逐字对比
model.generation_config.do_sample = False
# 采样关闭后 top_p 不再生效,显式设为 None 避免生成配置冲突警告
model.generation_config.top_p = None
# 同理,top_k 也显式设为 None
model.generation_config.top_k = None
# 显式指定 pad token id,避免 generate 时因未设置 pad_token_id 而报警告或使用错误的填充符
model.generation_config.pad_token_id = tokenizer.pad_token_id
# 将模型切换为评估(推理)模式,关闭 dropout 等训练专用行为;末尾分号用于抑制 Jupyter 单元格自动打印返回值
model.eval();

In [ ]:
# 构造符合 Gemma3 对话模板的消息列表,role 为 user,content 为上面定义的 prompt
messages = [{"role": "user", "content": prompt}]

# 使用分词器自带的 chat template 将消息转换为模型输入:
# tokenize=True 表示直接返回 token id 而非字符串;
# add_generation_prompt=True 会在末尾追加提示模型开始生成回复的特殊标记;
# return_tensors="pt" 返回 PyTorch 张量
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
)

# 调用官方模型的 generate 方法进行自回归文本生成,其结果作为与自实现 Gemma3 对比的"标准答案"
outputs = model.generate(
    **inputs,
    # 最多生成 500 个新 token,防止生成过长
    max_new_tokens=500,
    # 关闭采样,使用贪婪解码,保证结果确定、可复现
    do_sample=False,
    # 只保留 1 条候选序列(不使用 beam search),等价于贪婪搜索
    num_beams=1,
    # 显式传入 pad_token_id,避免因未设置而产生警告
    pad_token_id=tokenizer.pad_token_id,
)

# 将生成结果中的 token id 解码回文本;
# outputs[0][inputs["input_ids"].shape[-1]:] 只截取新生成的部分(去掉原始输入 prompt 对应的 token),
# skip_special_tokens=True 表示解码时跳过特殊标记(如结束符等)
response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)
# 打印官方 transformers 实现生成的参考回复文本,用于与自实现 Gemma3 的输出做逐字/逐 token 对比校验
print(response)